In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
for extra_path in (ROOT, ROOT / "models", ROOT / "src"):
    extra = str(extra_path)
    if extra not in sys.path:
        sys.path.append(extra)

from models.backbone import BackBone
from models.multiheadmodel import MultiHeadModel
from src.dataset import get_data_loaders
from src.utils import deterministic, accuracy


EXPERIMENT_MODE = "CIL"  # Change to "TIL" to test task-incremental learning.
CLASSES_PER_TASK = 2
EWC_LAMBDA = 1000.0


def local_labels(y, task_number, classes_per_task):
    return y - task_number * classes_per_task


def train_targets(y, task_number, classes_per_task, global_labels):
    return y if global_labels else local_labels(y, task_number, classes_per_task)


def freeze_head(head):
    for parameter in head.parameters():
        parameter.requires_grad = False


class SimpleEWC:
    def __init__(self, model, criterion, lambda_=1000.0, global_labels=False):
        self.model = model
        self.criterion = criterion
        self.lambda_ = lambda_
        self.global_labels = global_labels
        self.snapshots = []
        self.fishers = []

    def penalty(self):
        if not self.snapshots:
            return torch.tensor(0.0, device=next(self.model.parameters()).device)

        penalty = 0.0
        named_params = {name: param for name, param in self.model.named_parameters() if param.requires_grad}
        for fisher, snapshot in zip(self.fishers, self.snapshots):
            for name, param in named_params.items():
                if name not in fisher:
                    continue
                old_param = snapshot[name].to(param.device)
                fisher_diag = fisher[name].to(param.device)
                current = param
                if current.shape != old_param.shape:
                    current = current[: old_param.shape[0]]
                penalty = penalty + (fisher_diag * (current - old_param) ** 2).sum()
        return self.lambda_ * penalty

    def loss(self, logits, targets):
        return self.criterion(logits, targets) + self.penalty()

    @torch.no_grad()
    def _save_snapshot(self):
        self.snapshots.append({name: param.detach().clone() for name, param in self.model.named_parameters() if param.requires_grad})

    def update(self, dataloader, task_number, classes_per_task=2):
        device = next(self.model.parameters()).device
        named_params = {name: param for name, param in self.model.named_parameters() if param.requires_grad}
        fisher = {name: torch.zeros_like(param) for name, param in named_params.items()}

        self.model.eval()
        total_batches = max(1, len(dataloader))
        for x, y in dataloader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            self.model.zero_grad(set_to_none=True)
            logits = self.model(x, task_number)
            targets = train_targets(y, task_number, classes_per_task, self.global_labels)
            loss = self.criterion(logits, targets)
            loss.backward()

            for name, param in named_params.items():
                if param.grad is not None:
                    fisher[name] += (param.grad.detach() ** 2) / total_batches

        self.fishers.append(fisher)
        self._save_snapshot()
        self.model.train()


def train_task(model, train_loader, optimizer, ewc, task_number, epochs=5, classes_per_task=2, global_labels=False):
    device = next(model.parameters()).device
    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        progress = tqdm(train_loader, desc=f"Task {task_number} | Epoch {epoch + 1}/{epochs}", leave=False)
        for x, y in progress:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            targets = train_targets(y, task_number, classes_per_task, global_labels)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x, task_number)
            loss = ewc.loss(logits, targets)
            loss.backward()

            if global_labels and task_number > 0:
                old_classes = task_number * classes_per_task
                if model.heads["0"].weight.grad is not None:
                    model.heads["0"].weight.grad[:old_classes].zero_()
                if model.heads["0"].bias.grad is not None:
                    model.heads["0"].bias.grad[:old_classes].zero_()

            optimizer.step()

            running_loss += loss.item()
            progress.set_postfix(loss=loss.item())

        print(f"Task {task_number} epoch {epoch + 1}: loss={running_loss / max(1, len(train_loader)):.4f}")

In [6]:
deterministic(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataloaders = get_data_loaders(val_size=0.1, batch_size=128)

model = MultiHeadModel(BackBone()).to(device)
for parameter in model.backbone.parameters():
    parameter.requires_grad = True

criterion = nn.CrossEntropyLoss()
global_labels = EXPERIMENT_MODE.upper() == "CIL"
ewc = SimpleEWC(model, criterion, lambda_=EWC_LAMBDA, global_labels=global_labels)

for task_number, (train_loader, val_loader, test_loader) in enumerate(dataloaders):
    if global_labels:
        if task_number == 0:
            model.add_head(0, num_classes=CLASSES_PER_TASK)
        else:
            model.expand_head(0, num_class_increment=CLASSES_PER_TASK)
        head_number = 0
    else:
        model.add_head(task_number, num_classes=CLASSES_PER_TASK)
        head_number = task_number

    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    train_task(
        model,
        train_loader,
        optimizer,
        ewc,
        head_number,
        epochs=5,
        classes_per_task=CLASSES_PER_TASK,
        global_labels=global_labels,
    )
    ewc.update(train_loader, head_number, classes_per_task=CLASSES_PER_TASK)

    if not global_labels:
        freeze_head(model.heads[str(task_number)])

    print(f"\nTask {task_number} evaluation on all seen tasks ({EXPERIMENT_MODE.upper()}):")
    for eval_task in range(task_number + 1):
        eval_loader = dataloaders[eval_task][2]
        eval_head = 0 if global_labels else eval_task
        eval_acc = accuracy(model, eval_loader, eval_head, global_labels=global_labels)
        print(f"  Task {eval_task}: test_acc={eval_acc:.4f}")

Task [0, 1]: Train=9000, Val=1000, Test=2000
Task [2, 3]: Train=9000, Val=1000, Test=2000
Task [4, 5]: Train=9000, Val=1000, Test=2000
Task [6, 7]: Train=9000, Val=1000, Test=2000
Task [8, 9]: Train=9000, Val=1000, Test=2000


Task 0 | Epoch 1/5:   0%|          | 0/71 [00:00<?, ?it/s]

Task [0, 1]: Train=9000, Val=1000, Test=2000
Task [2, 3]: Train=9000, Val=1000, Test=2000
Task [4, 5]: Train=9000, Val=1000, Test=2000
Task [6, 7]: Train=9000, Val=1000, Test=2000
Task [8, 9]: Train=9000, Val=1000, Test=2000


Task 0 | Epoch 1/5:   0%|          | 0/71 [00:00<?, ?it/s]

Task 0 epoch 1: loss=0.3346


...

Task 0 epoch 3: loss=0.2705


Task 0 | Epoch 4/5:   0%|          | 0/71 [00:00<?, ?it/s]

Task 0 epoch 4: loss=0.2318


Task 0 | Epoch 5/5:   0%|          | 0/71 [00:00<?, ?it/s]

Task 0 epoch 5: loss=0.2201

Task 4 evaluation on all seen tasks (CIL):
  Task 0: test_acc=0.0000

Task 4 evaluation on all seen tasks (CIL):
  Task 0: test_acc=0.0000
  Task 1: test_acc=0.0000
  Task 2: test_acc=0.0000
  Task 1: test_acc=0.0000
  Task 2: test_acc=0.0000
  Task 3: test_acc=0.0000
  Task 4: test_acc=0.8895
  Task 3: test_acc=0.0000
  Task 4: test_acc=0.8895
